# 03. Classifier Stress Test

Run the classifier across all downloaded bills.
Goal: find `unknown` nodes and false positives to drive pattern improvements in `classify_bill.py`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

_repo = next(p for p in [Path().resolve(), *Path().resolve().parents] if (p / "pyproject.toml").exists())
_RESEARCH = _repo / "docs" / "research" / "financial-semantics"
sys.path.insert(0, str(_RESEARCH))

from classify_bill import DOLLAR, PRIMARY_LABELS, build_financial_df, check_coverage, classify_text  # noqa: E402

from deltatrack.bill_tree import normalize_bill  # noqa: E402

## Load and classify all bills

One version per bill (first downloaded). Adjust `xmls[0]` to target a specific version.

In [ ]:
BILLS_DIR = _repo / "bills"

rows = []
for bill_dir in sorted(BILLS_DIR.iterdir()):
    if not bill_dir.is_dir():
        continue
    xmls = sorted(bill_dir.glob("*.xml"))
    if not xmls:
        continue
    try:
        tree = normalize_bill(xmls[0])  # first version only
    except Exception as e:
        print(f"SKIP {bill_dir.name}: {e}")
        continue
    for n in tree.nodes:
        if not DOLLAR.search(n.body_text or ""):
            continue
        rows.append(
            {
                "bill": bill_dir.name,
                "label": classify_text(n.body_text),
                "path": " > ".join(n.display_path[-2:]) if n.display_path else "",
                "preview": (n.body_text or "")[:120],
                "body_text": n.body_text or "",
            }
        )

df = pd.DataFrame(rows)
print(f"{len(df)} dollar-amount nodes across {df['bill'].nunique()} bills")
df["label"].value_counts()

In [ ]:
for _, row in df[df["label"] == "cap"].iterrows():
    print(row["bill"], "|", row["path"])
    print()
    print(row["body_text"])
    print("---")

## Label distribution by bill

In [ ]:
df.pivot_table(
    index="bill",
    columns="label",
    values="path",
    aggfunc="count",
    fill_value=0,
)

## Coverage check

Verify that `build_financial_df` captures every dollar-amount node — no silent drops.

In [ ]:
BILLS_DIR = _repo / "bills"
for bill_dir in sorted(BILLS_DIR.iterdir()):
    if not bill_dir.is_dir():
        continue
    xmls = sorted(bill_dir.glob("*.xml"))
    if not xmls:
        continue
    try:
        tree = normalize_bill(xmls[0])
    except Exception as e:
        print(f"SKIP {bill_dir.name}: {e}")
        continue
    df_fin = build_financial_df(tree)
    print(f"{bill_dir.name}:", end=" ")
    check_coverage(df_fin, tree)

## Unknowns

These are the nodes that need new patterns in `classify_bill.py`.

In [ ]:
unknowns = df[df["label"] == "unknown"][["bill", "path", "preview", "body_text"]].reset_index(drop=True)
print(f"{len(unknowns)} unknown nodes")
unknowns[["bill", "path", "preview"]]

### Group by opening text

Frequent openers are the highest-priority patterns to add.

In [ ]:
# Group unknown nodes by opening text to find repeated patterns worth adding rules for
unknowns["opener"] = unknowns["body_text"].str[:80]
unknowns["opener"].value_counts().head(30)

### Inspect full text of a specific unknown

In [ ]:
# Read full text of a specific unknown row for pattern investigation
# Change idx to the row number you want to inspect
for _, row in unknowns.iloc[::5].iterrows():
    print(row["bill"], "|", row["path"])
    print()
    print(row["body_text"])
    print("---")

## False positive check

Spot-check `PRIMARY_LABELS` nodes — especially useful once non-appropriations bills are in the mix.

In [ ]:
# Spot-check PRIMARY_LABELS nodes in non-appropriations bills
# Look for nodes labelled appropriation/transfer/rescission that shouldn't be
primary = df[df["label"].isin(PRIMARY_LABELS)]
for _, row in primary.sample(min(15, len(primary)), random_state=42).iterrows():
    print(f"[{row['bill']}] [{row['label']}]")
    print(row["body_text"][:200])
    print()